In [ ]:
import pandas as pd

factor = 1
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# restrict part to only the columns and rows we need
sizes = [49, 14, 23, 45, 19, 3, 36, 9]
part_filtered = (
    part
    [(part.P_BRAND != "Brand#45")
     & (~part.P_TYPE.str.contains(r"^MEDIUM POLISHED"))
     & part.P_SIZE.isin(sizes)]
    [["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]]
)

# restrict partsupp to the columns we need
partsupp_filtered = partsupp[["PS_PARTKEY", "PS_SUPPKEY"]]

# inner‐join part and partsupp
total = (
    part_filtered
    .merge(partsupp_filtered,
           left_on="P_PARTKEY", right_on="PS_PARTKEY",
           how="inner")[["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]]
)

# build a deduplicated list of suppliers to exclude
supplier_keys = (
    supplier
    [supplier.S_COMMENT.str.contains(r"Customer(\S|\s)*Complaints")]
    ["S_SUPPKEY"].drop_duplicates()
)

# filter out any PS_SUPPKEY that appears in that supplier list
total = total[~total.PS_SUPPKEY.isin(supplier_keys)]

# group, count unique suppliers, rename, and sort
total = (
    total
    .groupby(["P_BRAND", "P_TYPE", "P_SIZE"],
              as_index=False, sort=False)
    .agg({"PS_SUPPKEY": "nunique"})
    .rename(columns={"PS_SUPPKEY": "SUPPLIER_CNT"})
    .sort_values(
        by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
        ascending=[False, True, True, True]
    )
)